# XLM-RoBERTa

In [5]:
import pandas as pd                                                                                                                                
import torch                                                                                                                                       
from transformers import AutoTokenizer, XLMRobertaForSequenceClassification                                                                        
from tqdm.auto import tqdm                                                                                                                         
                                                                                                                                            
# 1. Paths & Device                                                                                                                                
MODEL_PATH = r"D:\Ateneo de Davao\Thesis-Project-Aspect-Based-Sentiment-Analysis\_7_Model Fine-Tuning\xlmr_absa_weighted"                          
INPUT_CSV = r"D:\Ateneo de Davao\Thesis-Project-Aspect-Based-Sentiment-Analysis\Dataset\DB_master_reviews_combined.csv"                            
OUTPUT_CSV = r"D:\Ateneo de Davao\Thesis-Project-Aspect-Based-Sentiment-Analysis\Dataset\DB_master_reviews_with_sentiment.csv"   

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")  

In [6]:
df = pd.read_csv(INPUT_CSV)                                                                                                                        
df["sentiment"] = "no review_text"                                                                                                                 
                                                                                                                                                    
valid_mask = df["review_text"].notna() & (df["review_text"].astype(str).str.strip() != "")                                                         
valid_indices = df[valid_mask].index                                                                                                               
texts = df.loc[valid_indices, "review_text"].astype(str).tolist()                                                                                  
                                                                                                                                                     
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)                                                                                              
model = XLMRobertaForSequenceClassification.from_pretrained(MODEL_PATH).to(device)                                                                 
model.eval()                                                                                                                                       
                                                                                                                                                    
id2label = {0: "POSITIVE", 1: "NEUTRAL", 2: "NEGATIVE"}                                                                                            
batch_size = 32                                                                                                                                    
predictions = []                      

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [7]:
with torch.no_grad():                                                                                                                              
    for i in tqdm(range(0, len(texts), batch_size), desc="Inferring Sentiment"):                                                                   
        batch = texts[i : i + batch_size]                                                                                                          
        inputs = tokenizer(batch, padding=True, truncation=True, max_length=128, return_tensors="pt").to(device)                                   
        logits = model(**inputs).logits                                                                                                            
        preds = torch.argmax(logits, dim=-1).cpu().tolist()                                                                                        
        predictions.extend([id2label[p] for p in preds])                                                                                           
                                                                                                                                                                                                                                                   
df.loc[valid_indices, "sentiment"] = predictions                                                                                                   
df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")                                                                                           
                                                                                    
print("\nSentiment distribution:")                                                                                                                 
print(df["sentiment"].value_counts())    

Inferring Sentiment:   0%|          | 0/393 [00:00<?, ?it/s]


Sentiment distribution:
sentiment
POSITIVE          9278
no review_text    7015
NEGATIVE          3185
NEUTRAL            108
Name: count, dtype: int64


# Aspect Label Assignment

In [5]:
import re                                                                                                                                          
import pandas as pd                                                                                                                                
                                                                                                                                                
                                                                                                                                
INPUT_CSV = r"D:\Ateneo de Davao\Thesis-Project-Aspect-Based-Sentiment-Analysis\Dataset\DB_master_reviews_with_sentiment.csv"                            
OUTPUT_CSV = r"D:\Ateneo de Davao\Thesis-Project-Aspect-Based-Sentiment-Analysis\Dataset\DB_master_reviews_with_aspects.csv"                       
                                                                                                                                                
                                                                                                    
aspect_dictionary = {                                                                                                                              
    "Product Quality": [                                                                                                                           
        "coffee", "matcha", "latte", "espresso", "sikwate", "barako",                                                                              
        "pandesal", "pastry", "cake", "taste", "lami", "masarap", "lasa",                                                                          
        "pait", "tam-is", "umay", "lamiay", "food", "drinks", "isog",                                                                              
        "lasaw", "tab-ang", "luod", "puol", "init", "bugnaw", "delicious", "yummy", "tasty", "sweet", "drink", "tea", "americano",                                                                       
        "cappuccino", "frappe", "brew", "brewed", "hojicha", "pastries", "cakes",                                                                  
        "donut", "donuts", "doughnut", "doughnuts", "chicken", "pizza", "burger",                                                                  
        "pasta", "sandwich", "bread", "dessert", "desserts", "meal", "meals",                                                                      
        "foods", "rice", "beef", "breakfast", "brunch", "chocolate", "choco",                                                                      
        "creamy", "flavor", "flavour", "hot", "cold", "crispy", "fresh", "bland"                                                                   
    ],                                                                                                                                             
    "Customer Service": [                                                                                                                          
        "barista", "staff", "crew", "cashier", "service", "mabait", "sungit",                                                                      
        "accommodating", "suplada", "paspas", "dugay", "maayos", "attitude",                                                                       
        "hinay", "buotan", "snob", "taray", "friendly",                                                                                                                                                                                                            
        "staffs", "server", "servers", "waiter", "waiters", "waitress", "waitstaff",                                                               
        "order", "orders", "ordered", "serve", "served", "serving",                                                                                
        "welcoming", "polite", "attentive", "helpful", "fast", "slow",                                                                             
        "rude", "accommodate"                                                                                                                      
    ],                                                                                                                                             
    "Ambiance and Atmosphere": [                                                                                                                   
        "ambiance", "vibe", "aesthetic", "chill", "quiet", "noisy", "saba",                                                                        
        "sabaan", "mingaw", "music", "tugtog", "muni-muni", "chika", "tropa",                                                                      
        "atmosphere", "crowded", "samok", "ig-worthy", "instagrammable",                                                                           
        "ngitngit", "tambay",                                                                                                                                                                                                                            
        "ambience", "cozy", "cosy", "comfy", "vibes", "relaxing", "relax",                                                                         
        "peaceful", "calm", "view", "interior", "lighting", "decor",                                                                               
        "hangout", "tambayan", "tahimik", "muni muni"                                                                                              
    ],                                                                                                                                             
    "Facilities and Amenities": [                                                                                                                  
        "wifi", "internet", "connection", "socket", "outlet", "plug", "cr",                                                                        
        "restroom", "comfort room", "aircon", "tables", "chairs", "space",                                                                         
        "study", "tuon", "work", "laptop", "hina", "kusog", "tugnaw", "igang",                                                                                                                                                                             
        "ac", "air conditioning", "air conditioned", "clean", "cleanliness",                                                                       
        "seat", "seats", "seating", "comfort", "bathroom", "toilet", "washroom",                                                                   
        "sink", "hugaw", "dirty", "sockets", "plugs", "outlets"                                                                                    
    ],                                                                                                                                             
    "Price and Value": [                                                                                                                           
        "price", "value", "mahal", "barato", "sulit", "tipid", "overpriced",                                                                       
        "affordable", "sayang", "budget", "expensive", "cheap", "kamahal", "pricy",                                                                                                                                                                            
        "worth", "worth it", "presyo", "pricey", "cost", "pricing", "reasonable",                                                                  
        "portion", "portions", "serving size", "rates", "bill"                                                                                     
    ],                                                                                                                                             
    "Store Operations and Accessibility": [                                                                                                        
        "parking", "parkingan", "location", "accessible", "layo", "malayo",                                                                        
        "duol", "hours", "open", "close", "traffic", "tago",                                                                                                                                                                                         
        "drive-thru", "drive thru", "dine-in", "takeout", "take-out", "delivery",                                                                  
        "queue", "pila", "line"                                                                                                                    
    ]                                                                                                                                              
}      

In [6]:
compiled_patterns = {                                                                                                                              
    aspect: re.compile(                                                                                                                            
        r'\b(?:' + '|'.join(map(re.escape, sorted(keywords, key=len, reverse=True))) + r')\b',                                                     
        re.IGNORECASE                                                                                                                              
    )                                                                                                                                              
    for aspect, keywords in aspect_dictionary.items()                                                                                              
}                                                                                                                                                  
                                                                                                                                                    
def scan_aspects(text):                                                                                                                            
    # Check for empty, null, or whitespace-only review text                                                                                        
    if pd.isna(text) or not str(text).strip():                                                                                                     
        return "no review_text"                                                                                                                    
                                                                                                                                                    
    text_lower = str(text).lower()                                                                                                                 
    matched = [aspect for aspect, pattern in compiled_patterns.items() if pattern.search(text_lower)]                                              
                                                                                                                                                    
    return ", ".join(matched) if matched else "Unassigned"                                                                                         
                                                                                                                                                    
#Assign Aspect Labels                                                                                                         
df = pd.read_csv(INPUT_CSV)                                                                                                                        
df["aspect_labels"] = df["review_text"].apply(scan_aspects)                                                                                        
                                                                                                                                                
#Export to CSV                                                                                                                                 
df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")                                                                                           
                                                                                                                                                    
print(f"Saved dataset with aspect labels to: {OUTPUT_CSV}")                                                                                        
print("\n--- Aspect Assignment Summary ---")                                                                                                       
print(df["aspect_labels"].value_counts().head(10))

Saved dataset with aspect labels to: D:\Ateneo de Davao\Thesis-Project-Aspect-Based-Sentiment-Analysis\Dataset\DB_master_reviews_with_aspects.csv

--- Aspect Assignment Summary ---
aspect_labels
no review_text                                                7015
Product Quality                                               2526
Unassigned                                                    1641
Product Quality, Customer Service                             1347
Product Quality, Ambiance and Atmosphere                       885
Product Quality, Customer Service, Ambiance and Atmosphere     796
Customer Service                                               568
Ambiance and Atmosphere                                        522
Product Quality, Customer Service, Price and Value             327
Product Quality, Price and Value                               320
Name: count, dtype: int64
